# Phase 2 — LSE Computation

For each of the 60 meta-training datasets, runs all 6 pseudo-label generation methods,
applies Hungarian alignment, trains a fixed RF, and records LSE.

**Output**: `data/meta_table/meta_training.csv`  
Columns: `dataset_id | LSE_kmeans | LSE_dbscan | LSE_agg | LSE_gmm | LSE_autoenc | LSE_dictlearn | best_method`

**Rules (from CLAUDE.md)**:
- Random Forest: default sklearn params, `random_state=42`, never tuned
- `random_state=42` for all sklearn objects, `torch.manual_seed(42)` for PyTorch
- True labels seen only during Hungarian alignment and final evaluation
- Showcase datasets never in this table

In [15]:
import os, sys, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import openml

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT       = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR    = os.path.join(ROOT, 'data', 'raw')
META_DIR   = os.path.join(ROOT, 'data', 'meta_table')
MANIFEST   = os.path.join(META_DIR, 'dataset_manifest.csv')
CHECKPOINT = os.path.join(META_DIR, 'lse_checkpoint.csv')
OUTPUT     = os.path.join(META_DIR, 'meta_training.csv')

sys.path.insert(0, os.path.join(ROOT, 'src'))

openml.config.cache_directory = RAW_DIR

# ── Global seeds ──────────────────────────────────────────────────────────────
SEED = 42
import torch
np.random.seed(SEED)
torch.manual_seed(SEED)

print('Paths OK')

Paths OK


In [16]:
# ── Load manifest + showcase exclusion guard ──────────────────────────────────
SHOWCASE_IDS = {61, 187, 15, 53, 40966, 37, 54, 1590, 1597}

manifest = pd.read_csv(MANIFEST)
leaked = set(manifest['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'Showcase leak in manifest: {leaked}'

print(f'Manifest: {len(manifest)} datasets')
manifest.head()

Manifest: 104 datasets


,dataset_id,name,n_instances,n_features,n_classes
0,41004,jungle_chess_2pcs_endgame_lion_elephant,4704,46,3
1,4153,Smartphone-Based_Recognition_of_Human_Activities,180,66,6
2,1465,breast-tissue,106,9,6
3,119,"BNG(cmc,nominal,55296)",55296,9,3
4,26,nursery,12960,8,5


## Helper functions

In [17]:
# ── Preprocessing ─────────────────────────────────────────────────────────────

def load_and_split(dataset_id):
    """Download (cached), encode labels, 80/20 stratified split."""
    ds = openml.datasets.get_dataset(
        dataset_id,
        download_data=True,
        download_qualities=False,
        download_features_meta_data=False,
    )
    X, y, _, _ = ds.get_data(
        dataset_format='dataframe',
        target=ds.default_target_attribute,
    )
    # Drop any remaining non-numeric cols; coerce to float
    X = X.select_dtypes(include=[np.number]).astype(float)
    le = LabelEncoder()
    y_enc = le.fit_transform(y.astype(str))

    X_tr, X_te, y_tr, y_te = train_test_split(
        X.values, y_enc,
        test_size=0.2,
        random_state=SEED,
        stratify=y_enc,
    )
    return X_tr, X_te, y_tr, y_te


def scale(X_tr, X_te):
    """Fit StandardScaler on train, apply to both."""
    sc = StandardScaler()
    return sc.fit_transform(X_tr), sc.transform(X_te)

In [18]:
from lse import compute_lse, groundtruth_accuracy

## Clustering methods

In [19]:
from clustering import (
    pseudo_kmeans,
    pseudo_dbscan,
    pseudo_agglomerative,
    pseudo_gmm,
    pseudo_autoencoder,
    pseudo_dictlearn,
)

## Main loop with checkpointing

In [ ]:
METHODS = {
    'LSE_kmeans'   : pseudo_kmeans,
    'LSE_dbscan'   : pseudo_dbscan,
    'LSE_agg'      : pseudo_agglomerative,
    'LSE_gmm'      : pseudo_gmm,
    'LSE_autoenc'  : pseudo_autoencoder,
    'LSE_dictlearn': pseudo_dictlearn,
}

# Datasets excluded from the LSE loop.
# Size-based skips: clustering is prohibitively slow on these.
SKIP_IDS = {40685, 46536, 41166, 255, 119, 46955, 45548, 45927}

# Minimum ground-truth RF accuracy to include a dataset.
# When gt_acc is near random chance (e.g. 0.13 on a 10-class problem), the
# features don't predict the labels — LSE becomes unstable noise (LSE > 1.5
# is observed). Such datasets add no signal to the meta-learner.
MIN_GT_ACC = 0.30

# ── Resume from checkpoint if it exists ──────────────────────────────────────
if os.path.exists(CHECKPOINT):
    done_df  = pd.read_csv(CHECKPOINT)
    done_ids = set(done_df['dataset_id'])
    results  = done_df.to_dict('records')
    print(f'Resuming — {len(done_ids)} datasets already processed')
else:
    done_ids = set()
    results  = []
    print('Starting fresh')

all_diagnostics = []  # Collected only for datasets processed in THIS run.

# ── Main loop ─────────────────────────────────────────────────────────────────
total = len(manifest)

for i, row in manifest.iterrows():
    did   = int(row['dataset_id'])
    name  = row['name']
    n_cls = int(row['n_classes'])

    if did in done_ids:
        continue

    if did in SKIP_IDS:
        print(f'[{i+1:3d}/{total}] SKIP  id={did}  {name}  (in SKIP_IDS)')
        continue

    t0  = time.time()
    rec = {'dataset_id': did}

    try:
        X_tr, X_te, y_tr, y_te = load_and_split(did)

        if X_tr.shape[1] == 0:
            raise ValueError('No numeric features after loading')

        X_tr_sc, X_te_sc = scale(X_tr, X_te)

        gt_acc = groundtruth_accuracy(X_tr, y_tr, X_te, y_te)

        if gt_acc < MIN_GT_ACC:
            print(f'[{i+1:3d}/{total}] SKIP  id={did}  {name}  (gt_acc={gt_acc:.3f} < {MIN_GT_ACC} — LSE unstable)')
            done_ids.add(did)
            continue

        print(f'[{i+1:3d}/{total}] {name[:35]:35s}  gt={gt_acc:.3f}  n_cls={n_cls}')

        for col, fn in METHODS.items():
            rec[col] = float('nan')
            try:
                pseudo = fn(X_tr_sc, n_cls)
                lse, diag = compute_lse(
                    X_tr, y_tr, X_te, y_te, pseudo, gt_acc,
                    method_name=col.replace('LSE_', ''),
                    dataset_name=name,
                    verbose=True,
                )
                rec[col] = round(lse, 4)
                all_diagnostics.append(diag)
            except Exception as e:
                print(f'    {col} FAILED: {e}')
                all_diagnostics.append({
                    'dataset': name,
                    'method': col.replace('LSE_', ''),
                    'failed': True,
                    'error': str(e)[:200],
                })

        elapsed = time.time() - t0
        best    = max((col for col in METHODS if not np.isnan(rec.get(col, np.nan))),
                      key=lambda c: rec.get(c, -1))
        rec['best_method'] = best.replace('LSE_', '')
        rec['gt_accuracy'] = round(gt_acc, 4)

        results.append(rec)
        done_ids.add(did)
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False)

        print(f'           best={rec["best_method"]}  ({elapsed:.1f}s)')

    except Exception as e:
        print(f'[{i+1:3d}/{total}] FAIL  id={did}  {name}  — {e}')
        rec.update({c: float('nan') for c in METHODS})
        rec['best_method'] = 'FAILED'
        rec['gt_accuracy'] = float('nan')
        results.append(rec)
        done_ids.add(did)
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False)

print(f'\nDone. {len(results)} rows collected.')

# ── Inline diagnostics summary ────────────────────────────────────────────────
if all_diagnostics:
    _d = pd.DataFrame(all_diagnostics)
    print('\n=== Failure-mode frequency ===')
    for flag in ['cluster_collapse', 'cluster_degenerate', 'mapping_collapse',
                 'matches_majority', 'rf_underfit_pseudo']:
        if flag in _d.columns:
            n   = int(_d[flag].fillna(False).sum())
            pct = _d[flag].fillna(False).mean() * 100
            print(f'  {flag:25s}  {pct:5.1f}%  ({n} runs)')

    if 'matches_majority' in _d.columns:
        print('\n=== Datasets where EVERY method matches majority ===')
        bad      = _d.groupby('dataset')['matches_majority'].all()
        bad_list = bad[bad].index.tolist()
        print('  ' + ', '.join(bad_list) if bad_list else '  (none)')

    if 'lse_ratio' in _d.columns:
        print('\n=== Per-dataset LSE std across methods ===')
        print(_d.groupby('dataset')['lse_ratio'].std().describe().round(3).to_string())

## Build `meta_training.csv`

In [ ]:
# ── Assemble final table ──────────────────────────────────────────────────────
df = pd.DataFrame(results)

# Drop datasets where ALL methods failed
lse_cols = list(METHODS.keys())
all_nan  = df[lse_cols].isna().all(axis=1)
if all_nan.any():
    print(f'Dropping {all_nan.sum()} fully-failed datasets: {df.loc[all_nan, "dataset_id"].tolist()}')
    df = df[~all_nan].reset_index(drop=True)

# Drop datasets where gt_accuracy < MIN_GT_ACC (LSE is unstable noise there)
if 'gt_accuracy' in df.columns:
    low_gt = df['gt_accuracy'] < MIN_GT_ACC
    if low_gt.any():
        print(f'Dropping {low_gt.sum()} low-gt datasets (gt_acc < {MIN_GT_ACC}): {df.loc[low_gt, "dataset_id"].tolist()}')
        df = df[~low_gt].reset_index(drop=True)

# Recompute best_method from LSE columns (robust to partial failures)
def _best(row):
    vals = {c: row[c] for c in lse_cols if not np.isnan(row[c])}
    if not vals:
        return 'FAILED'
    return max(vals, key=vals.get).replace('LSE_', '')

df['best_method'] = df.apply(_best, axis=1)

# Column order
col_order = ['dataset_id'] + lse_cols + ['best_method', 'gt_accuracy']
df = df[col_order]

# Showcase exclusion final check
leaked = set(df['dataset_id']) & SHOWCASE_IDS
assert len(leaked) == 0, f'Showcase leak: {leaked}'

df.to_csv(OUTPUT, index=False)
print(f'Saved → {OUTPUT}')
print(f'Shape : {df.shape}')
df

In [22]:
# ── Write diagnostics CSV and print failure-mode summary ──────────────────────

DIAG_OUTPUT = os.path.join(META_DIR, 'diagnostics.csv')

if len(all_diagnostics) == 0:
    print('No diagnostics collected (likely all datasets resumed from checkpoint).')
    print('To collect diagnostics, delete lse_checkpoint.csv and rerun.')
else:
    diag_df = pd.DataFrame(all_diagnostics)
    diag_df.to_csv(DIAG_OUTPUT, index=False)
    print(f'Saved → {DIAG_OUTPUT}')
    print(f'Shape : {diag_df.shape}')

    print('\n=== Failure-mode frequency across all runs ===')
    flag_cols = ['cluster_collapse', 'cluster_degenerate', 'mapping_collapse',
                 'matches_majority', 'rf_underfit_pseudo']
    for flag in flag_cols:
        if flag in diag_df.columns:
            pct = diag_df[flag].fillna(False).mean() * 100
            print(f'  {flag:25s}  {pct:5.1f}%  ({int(diag_df[flag].fillna(False).sum())} runs)')

    if 'matches_majority' in diag_df.columns:
        print('\n=== Datasets where EVERY method matches majority (unclusterable) ===')
        bad = diag_df.groupby('dataset')['matches_majority'].all()
        bad_datasets = bad[bad].index.tolist()
        if bad_datasets:
            for d in bad_datasets:
                print(f'  {d}')
        else:
            print('  (none — good)')

    if 'lse_lift' in diag_df.columns:
        print('\n=== Top 15 datasets by max lse_lift across methods ===')
        top_lift = diag_df.groupby('dataset')['lse_lift'].max().sort_values(ascending=False).head(15)
        for d, v in top_lift.items():
            print(f'  {d:40s}  lse_lift={v:.3f}')

    if 'lse_ratio' in diag_df.columns:
        print('\n=== Per-dataset LSE variance across methods (meta-learnability signal) ===')
        variance = diag_df.groupby('dataset')['lse_ratio'].std().describe()
        print(variance.round(3).to_string())
        print('\n(If mean std is < 0.05, methods are too similar for a meta-learner to distinguish.)')

Saved → c:\MLResearch\data\meta_table\diagnostics.csv
Shape : (474, 21)

=== Failure-mode frequency across all runs ===
  cluster_collapse             8.9%  (42 runs)
  cluster_degenerate           8.9%  (42 runs)
  mapping_collapse             8.9%  (42 runs)
  matches_majority            12.4%  (59 runs)
  rf_underfit_pseudo           0.0%  (0 runs)

=== Datasets where EVERY method matches majority (unclusterable) ===
  (none — good)

=== Top 15 datasets by max lse_lift across methods ===
  CPMP-2015-runtime-classification          lse_lift=2.800
  visualizing_livestock                     lse_lift=1.667
  flags                                     lse_lift=1.500
  lymph                                     lse_lift=1.250
  cmc_seed_1_nrows_2000_nclasses_10_ncols_100_stratify_True  lse_lift=1.154
  cmc_seed_3_nrows_2000_nclasses_10_ncols_100_stratify_True  lse_lift=1.154
  cmc_seed_0_nrows_2000_nclasses_10_ncols_100_stratify_True  lse_lift=1.154
  JuanFeldmanIris                       

In [29]:
# ── Validation & summary ──────────────────────────────────────────────────────
# Always validate from the saved file, not in-memory df.
df = pd.read_csv(OUTPUT)
lse_cols = ['LSE_kmeans', 'LSE_dbscan', 'LSE_agg', 'LSE_gmm', 'LSE_autoenc', 'LSE_dictlearn']

print('=== LSE descriptive statistics ===')
print(df[lse_cols].describe().round(3).to_string())

print('\n=== Best-method distribution ===')
print(df['best_method'].value_counts())

print('\n=== NaN counts per method ===')
print(df[lse_cols].isna().sum())

# Sanity: LSE values should be in [0, ~1.5]
for col in lse_cols:
    valid = df[col].dropna()
    assert (valid >= 0).all(), f'{col} has negative LSE'
    assert (valid <= 1.5).all(), f'{col} has LSE > 1.5'

print('\nAll sanity checks passed.')
print(f'meta_training.csv ready for Phase 3 (meta-feature extraction).')

=== LSE descriptive statistics ===
       LSE_kmeans  LSE_dbscan  LSE_agg  LSE_gmm  LSE_autoenc  LSE_dictlearn
count      78.000      78.000   78.000   78.000       78.000         78.000
mean        0.618       0.593    0.615    0.653        0.621          0.540
std         0.217       0.301    0.198    0.229        0.220          0.184
min         0.212       0.103    0.222    0.242        0.252          0.236
25%         0.451       0.336    0.474    0.498        0.445          0.420
50%         0.602       0.564    0.602    0.646        0.606          0.507
75%         0.742       0.823    0.738    0.770        0.772          0.608
max         1.128       1.192    1.088    1.250        1.149          1.000

=== Best-method distribution ===
best_method
gmm          26
dbscan       21
kmeans       14
autoenc       9
agg           5
dictlearn     3
Name: count, dtype: int64

=== NaN counts per method ===
LSE_kmeans       0
LSE_dbscan       0
LSE_agg          0
LSE_gmm          0
LSE_au